In [ ]:
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np

from BaseModel import BuildModel
from HeatmapGen import plot_cortical_map_overlay

%load_ext autoreload
%autoreload 2

# Aging Tasks

In [ ]:
mech = {'FI', 'spine'}
A = BuildModel(age='Y', aging_mechanism=mech, J_max=0.468,G=0.21)
C = BuildModel(age="MA", aging_mechanism=mech, J_max=0.468,G=0.21)
D = BuildModel(age="A", aging_mechanism=mech, J_max=0.468,G=0.21)

# Fig 1
plt.figure(figsize=(12, 5))

plt.scatter(range(30), A.g_E_self, label='Young', alpha=0.7, s=50)
plt.scatter(range(30), C.g_E_self, label='Middle-aged', alpha=0.7, s=50)
plt.scatter(range(30), D.g_E_self, label='Aged', alpha=0.7, s=50)

plt.xlabel('Area')
plt.ylabel(fr'Local recurrent excitation')
plt.title(fr'Comparison of $J_i$ across age groups')
# plt.title(fr'Local recurrent excitation $\vec J$')

plt.legend()
plt.grid(alpha=0.3)
plt.xticks(range(C.num_areas), C.area_list, rotation=45)
plt.tight_layout()
plt.savefig(f'Fig/J_s_aging.svg', dpi=150, bbox_inches='tight')
plt.show()

# def phi_E_2(I, a_E, b_E, d_E):
#     return (a_E * I - b_E) / (1 - np.exp(-d_E * (a_E * I - b_E)))

# xc = np.linspace(0,1,100)
# for a,b,age in zip([135,140,140], [54,50,45],['Young', 'Middle Aged', 'Aged']):
#     plt.plot(xc,phi_E_2(xc, a, b, 0.308 ), label=fr"{age}")
# plt.title(fr'Transfer function $\phi_E$')
# plt.legend()
# plt.xlabel('Input current I (nA)')
# plt.ylabel('Firing rate (Hz)')
# plt.show()

## Aging and Lesion

In [ ]:
mech = {'FI','spine'}
B = BuildModel(J_max=0.26,G=0.48, aging_mechanism=mech,age='Y') 
selection = ['V1', 'MT', 'LIP', '24c', 'STPi', '9/46d'] # which plots to show

fig = plt.figure(figsize=(16,3*len(selection)), dpi= 80, facecolor='w', edgecolor='k')
trials=100

color_plot = { None: 'tab:blue',
              '24c': 'tab:orange',
              '9/46d': 'tab:green',
              }
              

for area in [None, '24c', '9/46d']: # areas to silence
    B.runsim(lesion = [ 'perm', area], stim=[4,4.5,0.6,'V1'], trials=trials) # silence the area here
    # B.plot_neural_activity()
    # plt.show()
    R_E1 = B.R[:,:B.num_areas]

    times = np.arange(4, 8, B.dt_rate)
    plt.rcParams.update({'font.size': 12})

    for i in range(1,len(selection)+1):
        ax = plt.subplot(len(selection),3,i)
        ax.set_title(selection[i-1])
        # Plot the rates for the E1 soma
        plt.subplots_adjust(hspace = 1)
        subset = np.where(np.isin(np.array(B.area_list), selection[i-1]))[0]
        for tr in range(trials):
            plt.plot(times, R_E1[int(4/B.dt_rate):int(8/B.dt_rate),subset,tr], color=color_plot[area],alpha = 0.5)        
        
        # Place text above the black line
        axes = plt.gca()
        if i == 1:
            # plt.legend(['Control', '24c silenced', '9/46d silenced'])
            legend_handles = [ mlines.Line2D([], [], color='tab:blue') ,
                               mlines.Line2D([], [], color='tab:orange'), 
                               mlines.Line2D([], [], color='tab:green') 
                               ]
            plt.legend(legend_handles, ['Control', '24c', '9/46d'])
        plt.xlabel('time (s)')
        plt.ylabel('firing rate (Hz)')
        plt.yticks(np.arange(0,np.max(R_E1[int(4/B.dt_rate):int(8/B.dt_rate),subset])+15, 10) )

fig.suptitle(f'Lesioning specific areas in a DWM network (Age: Y)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.savefig(f'Fig/Heatmaps/Lesion/example_lesion.svg', dpi=150, bbox_inches='tight')
plt.show()

### Heat maps

In [ ]:
# Inhibition condition applied to prefrontal areas during delay
# Heatmap_gen(mech={'FI','spine'}, n_jobs=-2, )


# Permanent lesion
# Heatmap_gen(mech={'FI','spine'}, n_jobs=-1, lesion=[True, '9/46d'])
# Heatmap_gen(mech={'FI','spine'}, n_jobs=-1, lesion=[True, '24c'])



In [ ]:

# Plot the lesion heatmaps
mech = {'FI','spine'}

# Plot_Heatmaps(mech=mech, metric='mean', lesion=[True, '9/46d'], condition_label='9/46d Lesion', diff=True)
# Plot_Heatmaps(mech=mech, metric='mean', lesion=[True, '9/46d'], condition_label='9/46d Lesion')
# Plot_Heatmaps(mech=mech, metric='mean', lesion=[True, '24c'], condition_label='24c Lesion', diff=True)
# Plot_Heatmaps(mech=mech, metric='mean', lesion=[True, '24c'], condition_label='24c Lesion')

In [ ]:
plot_cortical_map_overlay(mech=mech, J_max = 0.29,G = 0.48, lesion=[True, '9/46d'], condition_label='9/46d Lesion')
plot_cortical_map_overlay(mech=mech, J_max = 0.38,G = 0.46, lesion=[True, '9/46d'], condition_label='9/46d Lesion')

plot_cortical_map_overlay(mech=mech, J_max = 0.43,G = 0.44, lesion=[True, '24c'], condition_label='24c Lesion')


mech = {'FI','spine'}
j_maxes = [round(i * 0.01 + 0.21, 2) for i in range(31)]
Gs      = [round(j * 0.02 + 0.2,  2) for j in range(16)]

# for J in j_maxes:
#     for G in Gs:
#         plot_cortical_map_overlay(mech=mech, J_max=J,G=G, lesion=[True, '24c'], condition_label='24c Lesion')
# plot_cortical_map_overlay(mech=mech, J_max = 0.46,G = 0.46) this is how I store the pairs

## Aging and Inhibition

In [ ]:
# inhibtion example can be done with BuildModel directly

### Heatmaps

In [ ]:
# Inhibition
# Heatmap_gen(mech={'FI','spine'}, n_jobs=-1, inhi=[9, 9.5, 0.5, ['8m','9/46v','9/46d','8b']])
# Heatmap_gen(mech={'FI','spine'}, n_jobs=-1, inhi=[9, 9.5, 0.5, ['8m','9/46v','9/46d','8b']])

In [ ]:
# Plot_Heatmaps(mech=mech, metric='perc_act', inhi=[9, 9.5, 0.5, ['8m','9/46v','9/46d','8b']], condition_label='prefrontal inhibition')
# Plot_Heatmaps(mech=mech, metric='perc_nhall', inhi=[9, 9.5, 0.5, ['8m','9/46v','9/46d','8b']], condition_label='prefrontal inhibition')
# Plot_Heatmaps(mech=mech, metric='mean', inhi=[9, 9.5, 0.5, ['8m','9/46v','9/46d','8b']], condition_label='prefrontal inhibition')
# Plot_Heatmaps(mech=mech, metric='std', inhi=[9, 9.5, 0.5, ['8m','9/46v','9/46d','8b']], condition_label='prefrontal inhibition')

## Aging and Distractions

In [ ]:
h=0

mech = {'spine','FI'}
fig = plt.figure(figsize=(16,5), dpi= 80, facecolor='w', edgecolor='k')
for J_max, G in zip([0.468, 0.26 ],  [0.21, 0.48]):
    B = BuildModel(J_max=J_max,G=G)
    selection = ['V1', '9/46d'] # which plots to show
    B.runsim(stim=[2, 2.8, 0.3, 'V1'], trial_length = 15, distract=[6, 6.8, 0.3] )
    
    subset = np.where(np.isin(np.array(B.area_list), selection))[0]

    R_E1 = B.R[:,:B.num_areas,0]
    R_E2 = B.R[:,B.num_areas:2*B.num_areas,0]        

    times = np.arange(0, B.trial_length, B.dt_rate)
    plt.rcParams.update({'font.size': 12})

    for i in range(h+1,len(selection)+1+h):
        ax = plt.subplot(len(selection),4,i)
        ax.set_title(selection[(i-1)%2])
        # Plot the rates for the E1 soma
        plt.subplots_adjust(hspace = 1)
        subset = np.where(np.isin(np.array(B.area_list), selection[(i-1)%2]))[0]
        plt.plot(times, R_E1[:,subset], label="E1", color='tab:blue')        
        plt.plot(times, R_E2[:,subset], label="E2", color='tab:orange')  

        # axes = plt.gca()
        if i==1:
            plt.ylabel('firing rate (Hz)')
            plt.legend()

        plt.xlabel('time (s)')
        plt.yticks(np.arange(0,np.max(R_E1[:,subset])+10, 10))
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    fig.text(0.25, 1.02, "Resistance to distractors in LWM Regime", ha="center", fontsize=16)
    fig.text(0.75, 1.02, "Resistance to distractors in DWM Regime", ha="center", fontsize=16)
    h=2
plt.savefig(f'Fig/Heatmaps/Distractor/distractors_examples.svg', dpi=150, bbox_inches='tight')
plt.show()


# for J_max, G in zip([0.468, 0.26 ],  [0.21, 0.48]):
#     B = BuildModel(J_max=J_max,G=G)
#     B.runsim(stim=[2, 2.8, 0.3, 'V1'], trial_length = 15, distract=[ 6, 6.8, 0.3] )
#     B.raster(pops=2)


### Heatmaps

In [ ]:
# Distraction
# Heatmap_gen(mech={'FI','spine'}, n_jobs=-1, distract=[8,10,3])
# Heatmap_gen(mech={'FI','spine'}, n_jobs=-1, distract=[8,10,3])

In [ ]:
# Plot_Heatmaps(mech=mech, metric='resist',distract=[8,10,3])
# Plot_Heatmaps(mech=mech, metric='perc_act',distract=[8,10,3])
# Plot_Heatmaps(mech=mech, metric='perc_nhall',distract=[8,10,3])
# Plot_Heatmaps(mech=mech, metric='mean',distract=[8,10,3])
# Plot_Heatmaps(mech=mech, metric='std',distract=[8,10,3])